In [1]:
# input
tax_file = "./tmp/409_entryId-taxId.tsv"
align_pred_file = "./tmp/409_aligned_result.tsv"
# output
tree_file = "./tmp/tree.nwk"
tree_anno_file = "./tmp/tags_colorstrip.txt"

In [2]:
import pandas as pd

df_tax = pd.read_table(tax_file, header=None, names=["seq_id", "tax_id"])
df_align = pd.read_table(align_pred_file, usecols=["seq_id", "site_id"])
df = pd.merge(df_tax, df_align)

In [3]:
from ete4 import NCBITaxa
ncbi = NCBITaxa()

ranks = ncbi.get_rank(df['tax_id'].tolist())

def to_species_taxid(tid: int):
    if ranks.get(tid) == "species":
        return tid
    lineage = ncbi.get_lineage(tid)
    for anc in reversed(lineage):
        if ranks.get(anc) == "species":
            return anc
    return -1

In [4]:
df['sp_tax_id'] = df['tax_id'].map(lambda x: to_species_taxid(x))
df = df[df['sp_tax_id'] != -1]

/home/zhangf/install/miniforge3/envs/metalnet-helper/lib/python3.11/site-packages/ete4/ncbi_taxonomy/ncbiquery.py:219: UserWarning: taxid 759821 was translated into 253257
  warnings.warn('taxid %s was translated into %s' %


In [5]:
tax_id_to_label = dict()
for (tax_id,), df_tax in df.groupby(by=['tax_id']):
    site_ids = sorted(set(df_tax["site_id"]))
    has_site_1 = 1 in site_ids
    has_site_2 = 2 in site_ids
    if has_site_1 and has_site_2:
        label = "1,2"
    elif has_site_1 and not has_site_2:
        label = "1"
    elif not has_site_1 and has_site_2:
        label = "2"
    else:
        label = "0"
    
    if label != "0":
        tax_id_to_label[tax_id] = label

In [6]:
taxid_list = list(set(tax_id_to_label.keys()))
len(taxid_list)
species_tree = ncbi.get_topology(taxid_list)
len(species_tree)

354

352

In [7]:
names_dict = ncbi.get_taxid_translator(taxid_list)   # {taxid: 'Homo sapiens', …}
for leaf in species_tree.leaves():                   # ← 注意：ETE4 用 .leaves()
    sci = names_dict.get(leaf.taxid, f"taxid_{leaf.name}")
    leaf.name = sci.replace(" ", "_").split(":")[0]

for node in species_tree.traverse():
    if not node.is_leaf:
        node.name = ""
# 保存 Newick（给 iTOL 当树文件）
species_tree.write(outfile=tree_file)


all_labels = sorted(set(tax_id_to_label.values()))
all_labels
palette    = [
     "#8CA6D9", "#B3D9C7", "#EDBFD1",
]
color_map  = {lab: palette[i] for i, lab in enumerate(all_labels)}

# 头部
header = [
    "DATASET_COLORSTRIP",
    "SEPARATOR TAB",
    "DATASET_LABEL\tTag",
    "COLOR\t#000000",
    "LEGEND_TITLE\tTag",
    "LEGEND_SHAPES\t" + " ".join(["1"] * len(all_labels)),
    "LEGEND_COLORS\t" + " ".join(color_map[lab] for lab in all_labels),
    "LEGEND_LABELS\t" + " ".join(all_labels),
    "STRIP_WIDTH\t25",
    "MARGIN\t5",
    "DATA"
]

# 数据区
name_to_label = {names_dict[t].replace(" ", "_").split(":")[0]: tax_id_to_label[t]
                 for t in taxid_list}

data_lines = [f"{name}\t{color_map[label]}\t{label}"
              for name, label in name_to_label.items()]

with open(tree_anno_file, "w") as fo:
    fo.write("\n".join(header + data_lines))

print("✅  已输出：tree.nwk  &  tags_colorstrip.txt — 直接上传到 iTOL 即可。")

['1', '1,2', '2']

12381

✅  已输出：tree.nwk  &  tags_colorstrip.txt — 直接上传到 iTOL 即可。
